<a href="https://colab.research.google.com/github/bsuvathi06/tndrr-ai-assignment/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers datasets peft accelerate bitsandbytes

In [4]:
import json, torch, random
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

templates = [
    ("My payment failed twice on checkout", {"category": "billing", "priority": "high"}),
    ("How do I reset my account password?", {"category": "account", "priority": "low"}),
    ("App crashes immediately when opening reports", {"category": "bug", "priority": "high"}),
    ("Can you add dark mode to the dashboard?", {"category": "feature", "priority": "low"}),
    ("Refund not received after 10 days", {"category": "billing", "priority": "medium"}),
    ("Unable to update my registered email address", {"category": "account", "priority": "medium"}),
    ("Export button throws a 500 server error", {"category": "bug", "priority": "high"}),
    ("Need an invoice copy for last month", {"category": "billing", "priority": "low"}),
]
random.seed(42)
all_data = []
for i in range(300):
    text, label = random.choice(templates)
    all_data.append({"input": f"{text} (ticket #{1000+i})", "output": json.dumps(label)})

train_data, test_data = all_data[:250], all_data[250:]
model_id = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

def format_prompt(inp, out=None):
    p = f"Extract JSON with keys 'category' and 'priority' from request:\nRequest: {inp}\nJSON:"
    return f"{p} {out}{tokenizer.eos_token}" if out else p

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")

def evaluate(mdl, name):
    mdl.eval()
    valid_json, exact_match, failures = 0, 0, []
    for item in test_data:
        inputs = tokenizer(format_prompt(item["input"]), return_tensors="pt").to("cuda")
        with torch.no_grad():
            out_ids = mdl.generate(**inputs, max_new_tokens=40, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        gen = tokenizer.decode(out_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        try:
            parsed = json.loads(gen.split("\n")[0])
            valid_json += 1
            if parsed == json.loads(item["output"]):
                exact_match += 1
            else:
                failures.append((item["input"], item["output"], gen))
        except Exception:
            failures.append((item["input"], item["output"], gen))
    print(f"=== {name} === Valid JSON: {valid_json}/50 ({valid_json*2}%) | Exact Match: {exact_match}/50 ({exact_match*2}%)")
    return failures

evaluate(model, "BASE MODEL")

model = prepare_model_for_kbit_training(model)
lora_cfg = LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj", "v_proj"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM")
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

train_ds = Dataset.from_list([{"text": format_prompt(x["input"], x["output"])} for x in train_data])
tok_ds = train_ds.map(lambda e: tokenizer(e["text"], truncation=True, max_length=128), batched=True)

args = TrainingArguments(output_dir="./lora_out", per_device_train_batch_size=16, num_train_epochs=3, learning_rate=2e-4, fp16=True, logging_steps=15, save_strategy="no", report_to="none")
trainer = Trainer(model=model, args=args, train_dataset=tok_ds, data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False))
trainer.train()

ft_failures = evaluate(model, "FINE-TUNED MODEL")
print("Sample Failures:", ft_failures[:3])
import json, torch, random
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

templates = [
    ("My payment failed twice on checkout", {"category": "billing", "priority": "high"}),
    ("How do I reset my account password?", {"category": "account", "priority": "low"}),
    ("App crashes immediately when opening reports", {"category": "bug", "priority": "high"}),
    ("Can you add dark mode to the dashboard?", {"category": "feature", "priority": "low"}),
    ("Refund not received after 10 days", {"category": "billing", "priority": "medium"}),
    ("Unable to update my registered email address", {"category": "account", "priority": "medium"}),
    ("Export button throws a 500 server error", {"category": "bug", "priority": "high"}),
    ("Need an invoice copy for last month", {"category": "billing", "priority": "low"}),
]
random.seed(42)
all_data = []
for i in range(300):
    text, label = random.choice(templates)
    all_data.append({"input": f"{text} (ticket #{1000+i})", "output": json.dumps(label)})

train_data, test_data = all_data[:250], all_data[250:]
model_id = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

def format_prompt(inp, out=None):
    p = f"Extract JSON with keys 'category' and 'priority' from request:\nRequest: {inp}\nJSON:"
    return f"{p} {out}{tokenizer.eos_token}" if out else p

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")

def evaluate(mdl, name):
    mdl.eval()
    valid_json, exact_match, failures = 0, 0, []
    for item in test_data:
        inputs = tokenizer(format_prompt(item["input"]), return_tensors="pt").to("cuda")
        with torch.no_grad():
            out_ids = mdl.generate(**inputs, max_new_tokens=40, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        gen = tokenizer.decode(out_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        try:
            parsed = json.loads(gen.split("\n")[0])
            valid_json += 1
            if parsed == json.loads(item["output"]):
                exact_match += 1
            else:
                failures.append((item["input"], item["output"], gen))
        except Exception:
            failures.append((item["input"], item["output"], gen))
    print(f"=== {name} === Valid JSON: {valid_json}/50 ({valid_json*2}%) | Exact Match: {exact_match}/50 ({exact_match*2}%)")
    return failures

evaluate(model, "BASE MODEL")

model = prepare_model_for_kbit_training(model)
lora_cfg = LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj", "v_proj"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM")
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

train_ds = Dataset.from_list([{"text": format_prompt(x["input"], x["output"])} for x in train_data])
tok_ds = train_ds.map(lambda e: tokenizer(e["text"], truncation=True, max_length=128), batched=True)

args = TrainingArguments(output_dir="./lora_out", per_device_train_batch_size=16, num_train_epochs=3, learning_rate=2e-4, fp16=True, logging_steps=15, save_strategy="no", report_to="none")
trainer = Trainer(model=model, args=args, train_dataset=tok_ds, data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False))
trainer.train()

ft_failures = evaluate(model, "FINE-TUNED MODEL")
print("Sample Failures:", ft_failures[:3])


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

=== BASE MODEL === Valid JSON: 48/50 (96%) | Exact Match: 0/50 (0%)
trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


Map:   0%|          | 0/250 [00:00<?, ? examples/s]

/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1548: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
15,2.612603
30,1.703332
45,1.246874


=== FINE-TUNED MODEL === Valid JSON: 50/50 (100%) | Exact Match: 18/50 (36%)
Sample Failures: [('Refund not received after 10 days (ticket #1251)', '{"category": "billing", "priority": "medium"}', '{"category": "issue", "priority": "high"}'), ('Need an invoice copy for last month (ticket #1252)', '{"category": "billing", "priority": "low"}', '{"category": "invoice", "priority": "low"}'), ('Unable to update my registered email address (ticket #1253)', '{"category": "account", "priority": "medium"}', '{"category": "issue", "priority": "high"}')]


... (2 outputs left out)

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1548: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
15,2.612603
30,1.703332
45,1.246874


=== FINE-TUNED MODEL === Valid JSON: 50/50 (100%) | Exact Match: 18/50 (36%)
Sample Failures: [('Refund not received after 10 days (ticket #1251)', '{"category": "billing", "priority": "medium"}', '{"category": "issue", "priority": "high"}'), ('Need an invoice copy for last month (ticket #1252)', '{"category": "billing", "priority": "low"}', '{"category": "invoice", "priority": "low"}'), ('Unable to update my registered email address (ticket #1253)', '{"category": "account", "priority": "medium"}', '{"category": "issue", "priority": "high"}')]


In [5]:
!pip install -q pypdf sentence-transformers faiss-cpu requests

In [10]:
import faiss, numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

ds = load_dataset("CShorten/ML-ArXiv-Papers", split="train[:150]")
docs = [f"Title: {row['title']}\nAbstract: {row['abstract']}" for row in ds]
full_text = "\n\n".join(docs)
print("Total Characters:", len(full_text), "| Approx Pages:", round(len(full_text)/2500, 1))

def chunk_text(text, size, overlap):
    step = size - overlap
    return [text[i:i+size] for i in range(0, len(text), step)]

embedder = SentenceTransformer("all-MiniLM-L6-v2")

qa_pairs = []
for i in range(15):
    title = ds[i]["title"].strip().replace("\n", " ")
    abstract_snippet = ds[i]["abstract"].strip().replace("\n", " ")[:80]
    q = f"What is the main research focus and contribution of the paper titled '{title}'?"
    qa_pairs.append((q, abstract_snippet))

for size, overlap in [(300, 50), (900, 150)]:
    chunks = chunk_text(full_text, size, overlap)
    embs = np.array(embedder.encode(chunks, normalize_embeddings=True), dtype="float32")
    dim = embs.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embs)

    hits = 0
    failed_qs = []
    for q, expected_phrase in qa_pairs:
        q_emb = np.array(embedder.encode([q], normalize_embeddings=True), dtype="float32")
        _, I = index.search(q_emb, 5)
        top_indices = I[0]
        retrieved = " ".join([chunks[idx].lower() for idx in top_indices])
        if expected_phrase[:40].lower() in retrieved:
            hits += 1
        else:
            failed_qs.append(q)
    pct = round((hits / 15) * 100, 1)
    print("Chunk Size:", size, "| Total Chunks:", len(chunks), "| Recall@5:", hits, "/ 15 (", pct, "%)")
    if failed_qs:
        print("Sample Failed Q:", failed_qs[0])


Total Characters: 156213 | Approx Pages: 62.5


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Chunk Size: 300 | Total Chunks: 625 | Recall@5: 15 / 15 ( 100.0 %)
Chunk Size: 900 | Total Chunks: 209 | Recall@5: 14 / 15 ( 93.3 %)
Sample Failed Q: What is the main research focus and contribution of the paper titled 'Equivalence of LP Relaxation and Max-Product for Weighted Matching in   General Graphs'?


In [13]:
import time, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

prompts = [
    "Explain KV cache in transformers in two sentences.",
    "Write a Python function to check if a string is a palindrome.",
    "What is the capital of Australia and why was it chosen?",
    "Summarize the difference between TCP and UDP.",
    "Solve: If 3x + 7 = 22, what is x?",
    "List 3 benefits of database indexing.",
    "Translate 'Good morning, how are you?' into French.",
    "Explain overfitting to a 10-year-old.",
    "Write a SQL query to find duplicate emails in a Users table.",
    "Give one pro and one con of microservices."
]

def benchmark(quant_4bit=False):
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    if quant_4bit:
        bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
        mdl = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb, device_map="auto")
        label = "4-bit Quantised (NF4)"
    else:
        mdl = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16, device_map="auto")
        label = "16-bit (FP16)"

    total_tokens, total_time = 0, 0.0
    sample_outputs = []
    for p in prompts:
        inp = tokenizer(p, return_tensors="pt").to("cuda")
        in_len = inp["input_ids"].squeeze().numel()
        t0 = time.time()
        with torch.no_grad():
            out = mdl.generate(**inp, max_new_tokens=60, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        dt = time.time() - t0
        gen_ids = out.squeeze()[in_len:]
        total_tokens += gen_ids.numel()
        total_time += dt
        sample_outputs.append(tokenizer.decode(gen_ids, skip_special_tokens=True).strip())

    peak_mb = torch.cuda.max_memory_allocated() / (1024**2)
    tps = round(total_tokens / total_time, 2)
    print("===", label, "===")
    print("Speed:", tps, "tokens/sec | Peak GPU Memory:", round(peak_mb, 1), "MB")
    del mdl
    return sample_outputs

out_16 = benchmark(quant_4bit=False)
out_4 = benchmark(quant_4bit=True)
for i in range(10):
    print(f"\n--- Prompt {i+1}: {prompts[i]} ---")
    print("16-bit:", out_16[i].replace("\n", " ")[:140])
    print("4-bit :", out_4[i].replace("\n", " ")[:140])


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

=== 16-bit (FP16) ===
Speed: 29.55 tokens/sec | Peak GPU Memory: 7170.2 MB


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

=== 4-bit Quantised (NF4) ===
Speed: 14.74 tokens/sec | Peak GPU Memory: 5402.8 MB

--- Prompt 1: Explain KV cache in transformers in two sentences. ---
16-bit: In the context of transformer models, a key-value (KV) cache is used to store intermediate results and activations during training or infere
4-bit : The key-value (KV) cache is a type of memory used by transformers to store frequently accessed data, allowing for faster access and improved

--- Prompt 2: Write a Python function to check if a string is a palindrome. ---
16-bit: Sure, here's an example of a Python function that checks whether a given string is a palindrome:  ```python def is_palindrome(s):     """   
4-bit : The input string can contain any printable ASCII characters, including letters and digits.  **Function Signature:** ```python def is_palindr

--- Prompt 3: What is the capital of Australia and why was it chosen? ---
16-bit: The capital of Australia is Canberra. It was chosen because it was located in the cente